# 임베딩 모델 별 벡터스토어 저장 

In [ ]:
import os
import re
import pickle
import pandas as pd
import torch
from dotenv import load_dotenv
from langchain.vectorstores import FAISS
from langchain.schema import Document
from langchain.document_loaders import PDFPlumberLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, AutoTokenizer
from typing import List


# 환경 설정
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EXCLUDE_KEYWORDS = ['지원서', '사업계획서', '계획서', '서식', '양식', '신청서',
                    '동의서', '서약서', '의향서', '확인서']

In [ ]:
# 사용자 정의 임베딩 클래스
class KUREEmbedding:
    def __init__(self, model_name="nlpai-lab/KURE-v1"):
        self.model = SentenceTransformer(model_name, trust_remote_code=True).to(device)

    def embed_documents(self, texts):
        return self.model.encode(texts, convert_to_numpy=True)

    def embed_query(self, text):
        return self.embed_documents([text])[0]

class KoE5Embedding(KUREEmbedding):
    def __init__(self, model_name="nlpai-lab/KoE5"):
        super().__init__(model_name)

class KakaoEmbedding:
    def __init__(self, model_name="kakaobank/kf-deberta-base"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device)
    
    def embed_documents(self, texts):
        self.model.eval()
        embeddings = []
        with torch.no_grad():
            for text in texts:
                inputs = self.tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
                output = self.model(**inputs)
                cls_embedding = output.last_hidden_state[:, 0, :]  # [CLS] 토큰 임베딩 사용
                embeddings.append(cls_embedding.squeeze(0).cpu().numpy())
        return embeddings

    def embed_query(self, text):
        return self.embed_documents([text])[0]

In [ ]:
# 임베딩 모델
EMBEDDING_MODELS = {
    "openai": OpenAIEmbeddings(openai_api_key=openai_api_key),
    "koe5": KoE5Embedding(),
    "kure": KUREEmbedding(),
    "kf_deberta": KakaoEmbedding()
}

In [ ]:
# 줄바꿈 제거 함수
def remove_newlines_except_after_period(text):
    return re.sub(r'(?<!\.)(\n|\r\n)', ' ', text)

# 문서 로더
def load_document(file_path, folder_name, title, gov_name):
    ext = os.path.splitext(file_path)[1].lower()
    if ext == ".pdf":
        loader = PDFPlumberLoader(file_path)
    elif ext == ".json":
        loader = TextLoader(file_path, encoding="utf-8")
    else:
        return []

    docs = loader.load()
    for doc in docs:
        doc.page_content = remove_newlines_except_after_period(doc.page_content)
        doc.metadata.update({
            "folder_name": folder_name,
            "file_type": ext,
            "title": title,
            "소관부처_지자체": gov_name,
            "종류": "지원사업"
        })
    return docs

# 공통 저장 함수: 문서 리스트를 모델별로 벡터스토어 저장
def save_vectorstore_by_model(documents, base_vectorstore_dir):
    texts = [doc.page_content for doc in documents]
    metadatas = [doc.metadata for doc in documents]

    for model_name, model in EMBEDDING_MODELS.items():
        all_embeddings = []
        batch_size = 32
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            batch_embeddings = model.embed_documents(batch)
            all_embeddings.extend(batch_embeddings)

        vectorstore = FAISS.from_embeddings(list(zip(texts, all_embeddings)), model)
        save_dir = os.path.join(base_vectorstore_dir, f"embedding_{model_name}")
        os.makedirs(save_dir, exist_ok=True)
        vectorstore.save_local(save_dir)

        with open(os.path.join(save_dir, "documents.pkl"), "wb") as f:
            pickle.dump(documents, f)

# PDF/JSON 처리
def append_vectorstore_from_pdf_json(base_dir, vectorstore_dir, csv_path):
    df = pd.read_csv(csv_path)
    all_docs = []

    for folder_name in os.listdir(base_dir):
        folder_path = os.path.join(base_dir, folder_name)
        if not os.path.isdir(folder_path):
            continue

        match_row = df[df["pblanc_id"].astype(str) == folder_name]
        if match_row.empty:
            continue

        title = match_row.iloc[0]["제목"]
        gov_name = match_row.iloc[0].get("소관부처_지자체", "")

        for file_name in os.listdir(folder_path):
            if not (file_name.endswith(".pdf") or file_name.endswith(".json")):
                continue
            if any(keyword in file_name for keyword in EXCLUDE_KEYWORDS):
                continue

            file_path = os.path.join(folder_path, file_name)
            docs = load_document(file_path, folder_name, title, gov_name)
            all_docs.extend(docs)

    if not all_docs:
        return

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
    split_docs = splitter.split_documents(all_docs)

    texts = [doc.page_content for doc in split_docs]
    metadatas = [doc.metadata for doc in split_docs]

    for model_name, model in EMBEDDING_MODELS.items():

        embeddings = []
        batch_size = 32
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            batch_embeddings = model.embed_documents(batch)
            embeddings.extend(batch_embeddings)

        save_dir = os.path.join(vectorstore_dir, f"embedding_{model_name}")
        os.makedirs(save_dir, exist_ok=True)

        # 기존 인덱스 불러와서 추가 or 새로 생성
        if os.path.exists(os.path.join(save_dir, "index.faiss")):
            try:
                vectorstore = FAISS.load_local(save_dir, model.embed_query, allow_dangerous_deserialization=True)
                vectorstore.add_texts(texts=texts, metadatas=metadatas)
            except Exception as e:
                continue
        else:
            vectorstore = FAISS.from_embeddings(list(zip(texts, embeddings)), model)

        vectorstore.save_local(save_dir)

        # 문서 파일 저장 또는 병합
        doc_file = os.path.join(save_dir, "documents.pkl")
        try:
            if os.path.exists(doc_file):
                with open(doc_file, "rb") as f:
                    existing_docs = pickle.load(f)
                all_combined = existing_docs + split_docs
            else:
                all_combined = split_docs

            with open(doc_file, "wb") as f:
                pickle.dump(all_combined, f)

        except Exception as e:
            continue

def process_and_save_txt_documents(txt_dirs: List[str], vectorstore_dir: str):
    all_docs = []

    for txt_dir in txt_dirs:
        full_dir = os.path.join(os.getcwd(), txt_dir)
        if not os.path.exists(full_dir):
            continue

        for file_name in os.listdir(full_dir):
            if not file_name.endswith(".txt"):
                continue

            file_path = os.path.join(full_dir, file_name)
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    content = f.read()
            except Exception as e:
                continue

            clean_title = "_".join(file_name.split("_")[1:]).replace(".txt", "")
            metadata = {
                "folder_name": txt_dir,
                "file_type": ".txt",
                "title": clean_title,
                "종류": "금융상품"
            }

            doc = Document(page_content=remove_newlines_except_after_period(content), metadata=metadata)
            all_docs.append(doc)

    if not all_docs:
        return

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
    split_docs = splitter.split_documents(all_docs)

    texts = [doc.page_content for doc in split_docs]
    metadatas = [doc.metadata for doc in split_docs]

    for model_name, model in EMBEDDING_MODELS.items():

        save_dir = os.path.join(vectorstore_dir, f"embedding_{model_name}")
        os.makedirs(save_dir, exist_ok=True)

        # 배치로 한 번만 임베딩
        all_embeddings = []
        batch_size = 32
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            batch_embeddings = model.embed_documents(batch)
            all_embeddings.extend(batch_embeddings)

        try:
            if os.path.exists(os.path.join(save_dir, "index.faiss")):
                base_vs = FAISS.load_local(save_dir, model, allow_dangerous_deserialization=True)
                new_vs = FAISS.from_embeddings(list(zip(texts, all_embeddings)), model)
                base_vs.merge_from(new_vs)
                vectorstore = base_vs
            else:
                vectorstore = FAISS.from_embeddings(list(zip(texts, all_embeddings)), model)
        except Exception as e:
            continue

        # 저장
        vectorstore.save_local(save_dir)

        # 문서 병합 저장 (기존 + 신규)
        doc_file = os.path.join(save_dir, "documents.pkl")
        try:
            if os.path.exists(doc_file):
                with open(doc_file, "rb") as f:
                    existing_docs = pickle.load(f)
                all_combined = existing_docs + split_docs
            else:
                all_combined = split_docs

            with open(doc_file, "wb") as f:
                pickle.dump(all_combined, f)
        except Exception as e:
            continue

# 벡터스토어 생성

In [ ]:
append_vectorstore_from_pdf_json(
    base_dir="utils/data/download(support)",                 
    vectorstore_dir="embedding_comparison",  
    csv_path="utils/data/bizinfo_지원사업_공고목록.csv"
)

In [ ]:
VECTORSTORE_DIR = "embedding_comparison"
TXT_DIRS = ["utils/data/finance_products_txt"]
process_and_save_txt_documents(TXT_DIRS, VECTORSTORE_DIR)

### 벡터스토어 문서 수 확인

In [ ]:
def print_vectorstore_doc_counts(vectorstore_dir, embedding_models):
    for model_name, model in embedding_models.items():
        save_dir = os.path.join(vectorstore_dir, f"embedding_{model_name}")
        doc_file = os.path.join(save_dir, "documents.pkl")

        if not os.path.exists(save_dir):
            print(f"[{model_name}] 벡터스토어 디렉토리가 없습니다: {save_dir}")
            continue

        if os.path.exists(doc_file):
            try:
                with open(doc_file, "rb") as f:
                    docs = pickle.load(f)
                print(f"[{model_name}] 문서 개수: {len(docs)}")
            except Exception as e:
                print(f"[{model_name}] 문서 로드 실패: {e}")
        else:
            print(f"[{model_name}] documents.pkl이 없습니다.")

VECTORSTORE_DIR = "embedding_comparison"
print_vectorstore_doc_counts(VECTORSTORE_DIR, EMBEDDING_MODELS)

# 임베딩 모델 비교

In [ ]:
import os
import pickle
import random
from tqdm import tqdm
import numpy as np
import re
from langchain.vectorstores import FAISS
from langchain.schema import Document
from langchain.embeddings import OpenAIEmbeddings  
from langchain.chat_models import ChatOpenAI


# 설정
BASE_DIR = "embedding_comparison"
MODEL_DIRS = ["embedding_openai", "embedding_kure", "embedding_koe5", "embedding_kf_deberta"]
OPENAI_DIR = os.path.join(BASE_DIR, "embedding_openai")
DOC_PATH = os.path.join(OPENAI_DIR, "documents.pkl")

# GPT 질문 생성기
def generate_questions(chunk_text, llm):
    prompt = f"""
다음은 문서 일부입니다:
\"\"\"{chunk_text}\"\"\"

위 문서 내용을 바탕으로 이해도를 테스트할 수 있는 질문을 한국어로 5개 만들어주세요.
- 반드시 "1.", "2.", "3.", "4.", "5." 형식으로만 작성하세요.
- 불필요한 서두, 설명, 인사말 없이 질문만 출력하세요.
- 각 질문은 한 줄로 작성하세요.
"""
    response = llm.predict(prompt)

    # 번호로 시작하는 줄만 추출
    questions = []
    for line in response.split("\n"):
        line = line.strip()
        match = re.match(r"^\d+\.\s*(.*)", line)
        if match:
            questions.append(match.group(1).strip())

    return questions[:5]

# 청크 로딩 및 랜덤 샘플링
with open(DOC_PATH, "rb") as f:
    all_docs: list[Document] = pickle.load(f)

sampled_docs = random.sample(all_docs, 10)

# LLM 준비 (GPT로 질문 생성)
llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [ ]:
# 모델 객체 한 번만 로드
openai_model = OpenAIEmbeddings(openai_api_key=openai_api_key)
koe5_model = KoE5Embedding()
kure_model = KUREEmbedding()
kakao_model = KakaoEmbedding()

embedding_models = {
    "openai": openai_model,
    "koe5": koe5_model,
    "kure": kure_model,
    "kf_deberta": kakao_model
}

In [ ]:
results = {model_name: {k: [] for k in (3, 5, 10)} for model_name in embedding_models.keys()}

for run in range(30): 
    print(f"\n=== Run {run+1}/30 ===")

    # 사용할 문서 10개 샘플링
    sampled_docs = random.sample(all_docs, 10)

    # 질문 생성 + 메타데이터 출력
    doc_to_questions = []
    for i, doc in enumerate(tqdm(sampled_docs, desc="Generating questions")):
        chunk_text = doc.page_content[:1000]
        questions = generate_questions(chunk_text, llm)
        doc_to_questions.append((doc, questions))

        folder = doc.metadata.get("folder_name", "N/A")
        page = doc.metadata.get("page", "N/A")
        print(f"\n [샘플 {i+1}] folder_name: {folder} | page: {page}")
        print(f"문서 내용 일부:\n{chunk_text[:300]}...")
        print("생성된 질문:")
        for idx, q in enumerate(questions, 1):
            print(f"  {idx}. {q}")

    # 같은 질문셋으로 모든 모델 평가
    for model_name, embedding_obj in embedding_models.items():
        print(f"\nEvaluating: {model_name}")
        store_path = os.path.join(BASE_DIR, f"embedding_{model_name}")
        vectorstore = FAISS.load_local(store_path, embedding_obj.embed_query, allow_dangerous_deserialization=True)

        for k in (3, 5, 10):
            total = 0
            correct = 0
            for idx, (doc, questions) in enumerate(doc_to_questions, 1):
                for q_idx, q in enumerate(questions, 1):
                    print(f"[Run {run+1}/30] {model_name} | Recall@{k} | Doc {idx}/10 | Q{q_idx}/5", end="\r")
                    retrieved_docs = vectorstore.similarity_search(q, k=k)
                    retrieved_texts = [d.page_content.strip() for d in retrieved_docs]
                    if doc.page_content.strip() in retrieved_texts:
                        correct += 1
                    total += 1
            results[model_name][k].append(correct / total)

# 평균 & 표준오차 계산 → DataFrame 변환
final_rows = []
for model_name in embedding_models.keys():
    row = {"model": model_name}
    for k in (3, 5, 10):
        mean = np.mean(results[model_name][k])
        se = np.std(results[model_name][k], ddof=1) / np.sqrt(30)
        row[f"Recall@{k}"] = f"{mean:.4f} ({se:.4f})"
    final_rows.append(row)

df_results = pd.DataFrame(final_rows)
print("\n 최종 결과")
print(df_results.to_string(index=False))

In [ ]:
df_results.to_csv("임베딩결과.csv", index=False)